[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/masanorihirano/pams/blob/main/examples/dark_pool.ipynb)

In [ ]:
# Please remove comment-out if necessary
#! pip install pams matplotlib

# Dark pool

This example simulates a lit market and a dark pool beside it, following the artificial market model of Mizuta et al. (2015).
A lit market shows its order book to the traders, while a dark pool shows none.
The example first explains the model and the measures of the paper and how the classes below implement them.
It then runs a simulation and varies the usage of the dark pool to compare the results with the findings of the paper.

The same classes are in `samples/dark_pool` of the PAMS repository, and `samples/dark_pool/main.py` runs `samples/dark_pool/config.json` and prints the prices and the executed volume of each market at each step.

Reference:
- Mizuta, T., Kosugi, S., Kusumoto, T., Matsumoto, W., & Izumi, K. (2015). Effects of dark pools on financial markets' efficiency and price discovery function: an investigation by multi-agent simulations. *Evolutionary and Institutional Economics Review*, 12(2), 375–394. [https://doi.org/10.1007/s40844-015-0020-3](https://doi.org/10.1007/s40844-015-0020-3)

## Background

Dark pools are trading venues that publish neither their order books nor their quotes.
Their use has been growing, especially among institutional investors, because an investor can trade a large block of shares there without showing the order to others, and so with less market impact.
On the other hand, regulators are concerned that the trading moved away from the lit market may weaken its price discovery function and raise systemic risk.
When the paper was written, European regulators were discussing a cap on dark pools of 8% of the trading volume of each stock.

Empirical studies can hardly separate the effect of dark pools from the many other factors that move prices, and they cannot observe usage levels that have never occurred.
An artificial market can do both.
Mizuta et al. (2015) therefore use one to study how the share of the dark pool in trading affects the efficiency of the market and its price discovery function.
They add smart order routing (SOR), which sends an order that would trade at once to the dark pool when a counterpart is waiting there, to the model of an earlier study (Mizuta et al., 2015b, cited in the paper).
They also explain the mechanisms with a simple equation model.

## Model of Mizuta et al. (2015)

This section summarizes Sec. 2 of the paper.
The sections after it show how the classes of this example implement the model and where they differ from it.

### Order process

There are one risky asset, cash and $n$ agents.
Agent 1 orders at $t = 1$, agent 2 at $t = 2$, and so on, and after agent $n$ the cycle starts again from agent 1.
So exactly one order arrives at each step, and the time advances even if nothing is traded.
Every order is for one share, and the agents can sell short without limit.

### Lit market and dark pool

The lit market is a continuous double auction with the tick size $\delta P$.
Buy prices are rounded down to the tick, and sell prices are rounded up.
The paper calls an order a market order if it trades as soon as it arrives, and a limit order if it stays in the order book.
An order left in the order book is cancelled $t_c$ steps after it was placed.

The dark pool shows no order book, and an order sent to it has a side and one share but no price.
A new order trades at once if an opposite order is waiting in the dark pool, and waits there otherwise, so only buy orders or only sell orders can be waiting at any time.
The trade price is the mid price of the lit market,

$$P_{\mathrm{mid}} = \frac{P_{\mathrm{bid}} + P_{\mathrm{ask}}}{2},$$

where $P_{\mathrm{bid}}$ is the highest buy price and $P_{\mathrm{ask}}$ the lowest sell price in the order book of the lit market.
The paper notes that many real dark pools use this price.
The orders waiting in the dark pool are also cancelled after $t_c$ steps.

### Market selection

The paper studies two rules for choosing the market of each order.

- Without SOR, every order goes to the dark pool with the probability $d$ and to the lit market otherwise.
- With SOR, an order that would be a market order goes to the dark pool whenever opposite orders are waiting there.
  Every other order goes to the dark pool with the probability $d$, as without SOR.

The paper motivates SOR as follows.
A market order filled in the dark pool gets a better price than in the lit market by half the bid-ask spread.
Investors cannot see whether orders are waiting in a dark pool, but they are said to send a market order to a dark pool first and to send it to the lit market only if it does not trade there.

### Agents

Agent $j$ computes its expected return at time $t$ as

$$r^t_{e,j} = \frac{1}{w_{1,j} + w_{2,j} + u_j}\left(w_{1,j}\log\frac{P_f}{P^t} + w_{2,j}\,r^t_{h,j} + u_j\,\epsilon^t_j\right) \qquad (1)$$

and its expected price as

$$P^t_{e,j} = P^t\exp\left(r^t_{e,j}\right). \qquad (2)$$

- $P_f$ is the fundamental value, which is constant.
- $P^t$ is the market price.
  It keeps its last value at the steps without trades, and $P^1 = P_f$.
- The first term is the fundamental strategy, which expects the price to rise when it is below $P_f$ and to fall when it is above.
- The second term is the technical strategy, which follows the past return $r^t_{h,j} = \log(P^t / P^{t-\tau_j})$.
- The third term is the noise, where $\epsilon^t_j$ is normally distributed with the mean 0 and the size $\sigma_\epsilon$.
- The weights $w_{1,j}$, $w_{2,j}$ and $u_j$ and the time window $\tau_j$ are drawn once for each agent, uniformly from $(0, w_{1,\max})$, $(0, w_{2,\max})$, $(0, u_{\max})$ and $(1, \tau_{\max})$.

The order price $P^t_{o,j}$ is drawn from the normal distribution with the mean $P^t_{e,j}$ and the standard deviation $P_\sigma$.
The agent buys one share if $P^t_{e,j} > P^t_{o,j}$ and sells one share if $P^t_{e,j} < P^t_{o,j}$, so the side also follows from this draw.

### Parameters

Sec. 3 of the paper sets $n = 1{,}000$, $w_{1,\max} = 1$, $w_{2,\max} = 10$, $u_{\max} = 1$, $\tau_{\max} = 10{,}000$, $\sigma_\epsilon = 0.06$, $P_\sigma = 30$, $t_c = 20{,}000$, $\delta P = 0.1$ and $P_f = 10{,}000$, and runs each simulation to $t = 10^7$.
These values come from the earlier study, which chose them so that the model reproduces both stylized facts, such as fat tails and volatility clustering, and micro structures, such as execution and cancel rates.
With the lit market alone, the execution rate is 32.3% (Table 3 of the paper).
The Config section below lists these parameters beside those of this example.

In [ ]:
import copy
import random
from typing import Any
from typing import Dict
from typing import List
from typing import Optional
from typing import Tuple
from typing import Union
from typing import cast

import matplotlib.pyplot as plt
import numpy as np

from pams.agents import FCNAgent
from pams.logs import ExecutionLog
from pams.logs import MarketStepEndLog
from pams.logs import OrderLog
from pams.logs.market_step_loggers import MarketStepSaver
from pams.market import Market
from pams.order import MARKET_ORDER
from pams.order import Cancel
from pams.order import Order
from pams.runners import SequentialRunner
from pams.utils import JsonRandom

## Dark pool market

`DarkPoolMarket` is the dark pool of the paper.
Its lit market is set by the parameter `markets`.
It accepts only market orders, which have no price.
`SequentialRunner` executes the orders of a market after each order to it, so a new order trades at once if an opposite order is waiting and waits otherwise, and only one side can be waiting, as in the paper.
The orders are matched in time priority at the current mid price of the lit market, or at the market price of the lit market if one side of the order book of the lit market is empty.
The price is rounded to the tick size in favor of the earlier order.
A waiting order is cancelled when its time to live ends, which plays the role of $t_c$.
The dark pool cancels the orders placed while it is not running, that is, in the session without order execution.

In [ ]:
class DarkPoolMarket(Market):
    lit_market: Market

    def setup(self, settings: Dict[str, Any], *args: Any, **kwargs: Any) -> None:
        super().setup(settings, *args, **kwargs)
        if "markets" not in settings:
            raise ValueError("markets is required for DarkPoolMarket as the lit market")
        market_names: Any = settings["markets"]
        if not isinstance(market_names, list) or len(market_names) != 1:
            raise ValueError("markets of DarkPoolMarket must have exactly one market")
        if market_names[0] not in self.simulator.name2market:
            raise ValueError(f"lit market {market_names[0]} does not exist")
        lit_market: Market = self.simulator.name2market[market_names[0]]
        if isinstance(lit_market, DarkPoolMarket):
            raise ValueError("the lit market of DarkPoolMarket must not be a dark pool")
        self.lit_market = lit_market

    def get_lit_mid_price(self) -> float:
        best_buy_price: Optional[float] = self.lit_market.get_best_buy_price()
        best_sell_price: Optional[float] = self.lit_market.get_best_sell_price()
        if best_buy_price is None or best_sell_price is None:
            return self.lit_market.get_market_price()
        return (best_buy_price + best_sell_price) / 2.0

    def _add_order(self, order: Order) -> OrderLog:
        if order.kind != MARKET_ORDER:
            raise ValueError("DarkPoolMarket accepts only market orders")
        log: OrderLog = super()._add_order(order=order)
        if not self.is_running:
            self._cancel_order(cancel=Cancel(order=order))
        return log

    def remain_executable_orders(self) -> bool:
        return len(self.buy_order_book) > 0 and len(self.sell_order_book) > 0

    def _execution(self) -> List[ExecutionLog]:
        logs: List[ExecutionLog] = []
        lit_mid_price: float = self.get_lit_mid_price()
        while self.remain_executable_orders():
            buy_order: Order = cast(Order, self.buy_order_book.get_best_order())
            sell_order: Order = cast(Order, self.sell_order_book.get_best_order())
            is_buy_later: bool = cast(int, buy_order.order_id) > cast(int, sell_order.order_id)
            # rounding as a sell price (up) favors the earlier sell order and rounding
            # as a buy price (down) favors the earlier buy order.
            price: float = self.convert_to_price(
                tick_level=self.convert_to_tick_level(price=lit_mid_price, is_buy=not is_buy_later)
            )
            volume: int = min(buy_order.volume, sell_order.volume)
            logs.append(
                self._execute_orders(price=price, volume=volume, buy_order=buy_order, sell_order=sell_order)
            )
        return logs

## Dark pool FCN agent

`DarkPoolFCNAgent` implements the market selection without SOR.
It orders only when it is asked for the orders to a dark pool.
It then decides its order to the lit market of the dark pool as `FCNAgent` does, and with the probability `darkPoolChance`, which is $d$ of the paper, it sends the order to the dark pool instead, as a market order with the same side, volume and time to live.
This choice does not depend on the orders waiting in the dark pool, so the agent does not use SOR.

`FCNAgent` of PAMS is close to the agent of Eqs. (1) and (2).
Let $F$, $C$ and $N$ be its `fundamentalWeight`, `chartWeight` and `noiseWeight`, $\tau$ its `timeWindowSize` and $\sigma_n$ its `noiseScale`.
When `meanReversionTime` takes its default value $\tau$ and $t \geq \tau$, its expected price is

$$P_e = P^t\exp\left(\frac{F\log(P_f / P^t) + C\log(P^t / P^{t-\tau}) + N\tau\sigma_n\epsilon}{F + C + N}\right),$$

where $\epsilon$ is standard normal.
This is Eqs. (1) and (2) with $w_{1,j} = F$, $w_{2,j} = C$ and $u_j = N$, except that the noise is multiplied by $\tau$.
Unlike the paper, the agent buys if $P_e$ is above the market price $P^t$ and sells if it is below, at the price $P_e(1 - m)$ for a buy and $P_e(1 + m)$ for a sell, where $m$ is its `orderMargin`.
Its orders are limit orders with the time to live $\tau$.
In the terms of the paper, such an order is a market order if it trades on arrival and a limit order otherwise.

In [ ]:
class DarkPoolFCNAgent(FCNAgent):
    dark_pool_chance: float

    def setup(
        self, settings: Dict[str, Any], accessible_markets_ids: List[int], *args: Any, **kwargs: Any
    ) -> None:
        super().setup(settings=settings, accessible_markets_ids=accessible_markets_ids)
        if "darkPoolChance" not in settings:
            raise ValueError("darkPoolChance is required for DarkPoolFCNAgent")
        self.dark_pool_chance = JsonRandom(prng=self.prng).random(json_value=settings["darkPoolChance"])
        if not 0.0 <= self.dark_pool_chance <= 1.0:
            raise ValueError("darkPoolChance must be between 0.0 and 1.0")

    def submit_orders_by_market(self, market: Market) -> List[Union[Order, Cancel]]:
        if not isinstance(market, DarkPoolMarket):
            return []
        if not self.is_market_accessible(market_id=market.market_id):
            return []
        orders: List[Union[Order, Cancel]] = []
        for lit_order in super().submit_orders_by_market(market=market.lit_market):
            if isinstance(lit_order, Order) and self.prng.random() < self.dark_pool_chance:
                orders.append(
                    Order(
                        agent_id=self.agent_id,
                        market_id=market.market_id,
                        is_buy=lit_order.is_buy,
                        kind=MARKET_ORDER,
                        volume=lit_order.volume,
                        ttl=lit_order.ttl,
                    )
                )
            else:
                orders.append(lit_order)
        return orders

## Saver

The saver records the executed volume and the executed value, that is, the sum of the volume times the price, of each market at each step.
It also counts the orders placed to each market at each step.
The measures of the paper below use these records.

In [ ]:
class DarkPoolMarketStepSaver(MarketStepSaver):
    def __init__(self) -> None:
        super().__init__()
        self.n_orders: Dict[Tuple[int, int], int] = {}

    def process_order_log(self, log: OrderLog) -> None:
        key = (log.market_id, log.time)
        self.n_orders[key] = self.n_orders.get(key, 0) + 1

    def process_market_step_end_log(self, log: MarketStepEndLog) -> None:
        super().process_market_step_end_log(log=log)
        self.market_step_logs[-1]["executed_volume"] = log.market.get_executed_volume()
        self.market_step_logs[-1]["executed_value"] = log.market.get_executed_total_price()

## Config

The simulation has two sessions.
In session 0 (100 steps), the agents place orders but no order is executed, so that the order book of the lit market is filled first.
In session 1 (500 steps), the orders are executed.
`maxNormalOrders` is 2, the number of markets, as the `MEMO` says, so two randomly chosen agents order at each step and each of them sends one order.
The default of PAMS is 1, and the last section explains where the value 2 comes from.
In the paper, one agent orders at each step in a fixed cycle instead.

The table compares the parameters of the paper with this config.

| Paper | Config | Value in the paper (Sec. 3) | Value in this example |
|---|---|---|---|
| $n$ | `numAgents` | 1,000 | 100 |
| $P_f$ and $P^1$ | `marketPrice` | 10,000 | 300 |
| $\delta P$ | `tickSize` | 0.1 | 0.00001 |
| $w_{1,j}$ | `fundamentalWeight` | uniform on (0, 1) | exponential with the mean 1 |
| $w_{2,j}$ | `chartWeight` | uniform on (0, 10) | 0 |
| $u_j$ | `noiseWeight` | uniform on (0, 1) | exponential with the mean 1 |
| $\sigma_\epsilon$ | `noiseScale` times $\tau$ | 0.06 | 0.001 times $\tau$ |
| $\tau_j$ | `timeWindowSize` | uniform on (1, 10,000) | an integer from 100 to 199 |
| $P_\sigma$ | `orderMargin` (a different rule) | 30 | uniform on [0, 0.1) |
| $t_c$ | `timeWindowSize` (the time to live) | 20,000 | 100 to 199 |
| $d$ | `darkPoolChance` | 0 to 0.9 | 0.3 |
| $t_e$ | `iterationSteps` of session 1 | $10^7$ | 500 |

The fundamental price starts at `marketPrice` and stays constant, because `fundamentalVolatility` is not set and defaults to 0.
`chartWeight` is always 0, so the agents of this example have no technical term, while in the paper the weight of this term is drawn from a range ten times as wide as those of the other two terms.
`assetVolume` and `cashAmount` are the initial holdings of each agent.
The dark pool has the same tick size and initial price as the lit market, and its lit market is `LitMarket`.
SOR is not implemented, so this example corresponds to the case without SOR in the paper.

In [ ]:
config = {
	"simulation": {
		"markets": ["LitMarket", "DarkPoolMarket"],
		"agents": ["DarkPoolFCNAgents"],
		"sessions": [
			{	"sessionName": 0,
				"iterationSteps": 100,
				"withOrderPlacement": True,
				"withOrderExecution": False,
				"withPrint": True,
				"maxNormalOrders": 2, "MEMO": "The same number as #markets"
			},
			{	"sessionName": 1,
				"iterationSteps": 500,
				"withOrderPlacement": True,
				"withOrderExecution": True,
				"withPrint": True,
				"maxNormalOrders": 2, "MEMO": "The same number as #markets"
			}
		]
	},

	"LitMarket": {
		"class": "Market",
		"tickSize": 0.00001,
		"marketPrice": 300.0,
		"outstandingShares": 25000
	},

	"DarkPoolMarket": {
		"class": "DarkPoolMarket",
		"tickSize": 0.00001,
		"marketPrice": 300.0,
		"outstandingShares": 25000,
		"markets": ["LitMarket"]
	},

	"DarkPoolFCNAgents": {
		"class": "DarkPoolFCNAgent",
		"numAgents": 100,

		"MEMO": "Agent class",
		"markets": ["LitMarket", "DarkPoolMarket"],
		"assetVolume": 50,
		"cashAmount": 10000,

		"MEMO": "FCNAgent class",
		"fundamentalWeight": {"expon": [1.0]},
		"chartWeight": {"expon": [0.0]},
		"noiseWeight": {"expon": [1.0]},
		"noiseScale": 0.001,
		"timeWindowSize": [100, 200],
		"orderMargin": [0.0, 0.1],

		"MEMO": "DarkPoolFCNAgent class",
		"darkPoolChance": 0.3
	}
}

## Simulation

`run` runs a simulation and returns the time series of each market: the market price, the fundamental price, the executed volume, the executed value and the number of orders at each step.
It also returns the trade price at each step, which is the market price of the dark pool if the dark pool executed some orders at the step, otherwise that of the lit market if the lit market did, and otherwise NaN.

In [ ]:
def run(config: Dict[str, Any], seed: int) -> Dict[str, np.ndarray]:
    saver = DarkPoolMarketStepSaver()
    runner = SequentialRunner(settings=config, prng=random.Random(seed), logger=saver)
    runner.class_register(cls=DarkPoolMarket)
    runner.class_register(cls=DarkPoolFCNAgent)
    runner.main()

    result: Dict[str, np.ndarray] = {}
    for market_name, prefix in [("LitMarket", "lit"), ("DarkPoolMarket", "dark")]:
        logs = sorted(
            filter(lambda x: x["market_name"] == market_name, saver.market_step_logs),
            key=lambda x: x["market_time"],
        )
        result["time"] = np.array([x["market_time"] for x in logs])
        result["session_id"] = np.array([x["session_id"] for x in logs])
        result[f"{prefix}_price"] = np.array([x["market_price"] for x in logs], dtype=float)
        result[f"{prefix}_fundamental"] = np.array([x["fundamental_price"] for x in logs], dtype=float)
        result[f"{prefix}_volume"] = np.array([x["executed_volume"] for x in logs], dtype=int)
        result[f"{prefix}_value"] = np.array([x["executed_value"] for x in logs], dtype=float)
        result[f"{prefix}_orders"] = np.array(
            [saver.n_orders.get((x["market_id"], x["market_time"]), 0) for x in logs], dtype=int
        )
    result["trade_price"] = np.where(
        result["dark_volume"] > 0,
        result["dark_price"],
        np.where(result["lit_volume"] > 0, result["lit_price"], np.nan),
    )
    return result

In [ ]:
result = run(config=config, seed=42)

## Prices

The plot shows the market prices of both markets, the fundamental price and the trade price.
The market price of the lit market is $P^t$ of the paper.
The trade price keeps its last value at the steps without executions, and starts from the initial market price, 300.
In the session without order execution, the market prices do not change.

In [ ]:
trade_price_filled = np.empty_like(result["trade_price"])
last_trade_price = 300.0
for i, trade_price in enumerate(result["trade_price"]):
    if not np.isnan(trade_price):
        last_trade_price = trade_price
    trade_price_filled[i] = last_trade_price

plt.plot(result["time"], result["lit_price"], color="gold", label="lit market price")
plt.plot(result["time"], result["dark_price"], color="blue", label="dark pool price")
plt.plot(result["time"], result["lit_fundamental"], color="black", label="fundamental price")
plt.plot(result["time"], trade_price_filled, color="red", label="trade price")
plt.ylim(280, 320)
plt.xlabel("t")
plt.ylabel("Price")
plt.legend()
plt.show()

## Trade volumes

The plot shows the centered moving sum of the executed volume of each market over 100 steps.
At the first step of session 1, many orders placed in session 0 are executed at once in the lit market, so the moving sum of the lit market is higher while its window contains that step, from about t = 50 to t = 150.

In [ ]:
window = 100
moving_sum_time = result["time"][window // 2 - 1 : len(result["time"]) - window // 2]
plt.plot(
    moving_sum_time,
    np.convolve(result["lit_volume"], np.ones(window), mode="valid"),
    color="gold",
    label="lit market",
)
plt.plot(
    moving_sum_time,
    np.convolve(result["dark_volume"], np.ones(window), mode="valid"),
    color="blue",
    label="dark pool",
)
plt.xlim(0, len(result["time"]))
plt.ylim(0, 100)
plt.xlabel("t")
plt.ylabel("Volume")
plt.legend()
plt.show()

## Measures

`summarize` computes the following measures of the paper from the result of `run`.

The market inefficiency, Eq. (3) of the paper, is the mean relative distance of the market price from the fundamental price,

$$M_{ie} = \frac{1}{t_e}\sum_{t=1}^{t_e}\frac{\left|P^t - P_f\right|}{P_f}. \qquad (3)$$

It is 0 when the market is perfectly efficient and grows as the market becomes less efficient.
It needs the fundamental price, so it can be computed in simulations and experiments but not from real market data (footnote 4 of the paper).
`market_inefficiency` is $M_{ie}$ of the lit market in percent over session 1.

The share of the dark pool is the share of the traded value in the dark pool,

$$D = \frac{V_D}{V_D + V_L},$$

where $V_D$ and $V_L$ are the traded values in the dark pool and in the lit market.
`dark_share` is $D$, and `lit_volume` and `dark_volume` are the executed volumes.
$D$ is not the probability $d$ of sending an order to the dark pool.
Let $k$ be the probability that an order to the lit market is a market order, that is, trades on arrival.
Since few orders in the dark pool are cancelled in its simulations, the paper assumes that half of the orders to the dark pool wait and the other half trade on arrival, and derives

$$D = \frac{d/2}{k(1 - d) + d/2}. \qquad (4)$$

As every order is for one share, $D$ is then the share of the dark pool in the orders that trade on arrival (footnote 6 of the paper).
Eq. (4) gives $D > d$ whenever $k < 1/2$.

The execution rate is the number of trades divided by the number of orders (footnote 8 of the paper).
The paper also denotes the execution rate of the lit market by $k$ (Fig. 2).
`lit_execution_rate` is the execution rate of the lit market.
It counts the orders of both sessions, because the orders placed in session 0 can trade at the start of session 1.

In [ ]:
def summarize(result: Dict[str, np.ndarray]) -> Dict[str, float]:
    in_session = result["session_id"] == 1
    lit_value = float(result["lit_value"].sum())
    dark_value = float(result["dark_value"].sum())
    lit_prices = result["lit_price"][in_session]
    fundamental_prices = result["lit_fundamental"][in_session]
    # every order is for one share, so the executed volume is the number of trades.
    lit_volume = int(result["lit_volume"].sum())
    return {
        "lit_volume": lit_volume,
        "dark_volume": int(result["dark_volume"].sum()),
        "dark_share": dark_value / (lit_value + dark_value) if lit_value + dark_value > 0 else 0.0,
        "market_inefficiency": 100 * float(np.mean(np.abs(lit_prices - fundamental_prices) / fundamental_prices)),
        "lit_execution_rate": lit_volume / max(int(result["lit_orders"].sum()), 1),
    }


for key, value in summarize(result).items():
    print(f"{key}: {value:.4g}")

## Findings of Mizuta et al. (2015)

### Design of the experiments

The paper runs the simulation for 17 values of $d$ from 0 to 0.9: every 0.025 up to 0.2, then 0.25 and 0.3, and then every 0.1 up to 0.9.
All the other parameters and the random numbers are the same for every value of $d$, and the whole set is repeated 100 times with different random numbers.
The results are averages over the 100 runs and are plotted against the share $D$, not against $d$.

### Without SOR

$M_{ie}$ decreases gradually and monotonically as $D$ grows (Fig. 1).
The execution rate of the lit market is stable and decreases only moderately (Fig. 2).

### With SOR

$M_{ie}$ decreases more sharply than without SOR up to about $D = 70\%$, and increases sharply above it (Fig. 1).
So there is an optimal usage of the dark pool for the market efficiency.

The paper explains the decrease as follows (Figs. 2 to 5).
SOR sends market orders to the dark pool when opposite orders are waiting there, so the execution rate of the lit market falls faster than without SOR.
This raises the share of limit orders among the orders to the lit market, so its order book becomes deeper and its bid-ask spread narrower (Fig. 4).
The deeper order book absorbs the market orders that would move the price away from $P_f$, so the price stays near $P_f$.

The paper explains the increase as follows (Figs. 6 and 7).
When $D$ is high and $P^t$ is far above $P_f$, many buy orders are waiting in the dark pool.
The agents then expect the price to fall and send market sell orders, which would bring $P^t$ back to $P_f$.
SOR, however, sends these orders to the dark pool, where the waiting buy orders absorb them, so $P^t$ stays far above $P_f$.
The same happens with buy and sell exchanged when $P^t$ is far below $P_f$.

### Equation model

Sec. 4 of the paper explains these results with the probabilities of each kind of order.
Let $a$ be the probability that an order is a buy order.
With $a = 1/2$, SOR reduces the execution rate of the lit market from $k$ to

$$k' = \left(1 - \frac{1}{3}\,\frac{D}{1 - D}\right)k \quad \text{if } D < \frac{1}{2}, \qquad (9)$$

$$k' = \frac{2}{3}\,k \quad \text{if } D > \frac{1}{2}. \qquad (10)$$

The factor 1/3 assumes that the dark pool is equally likely to have buy orders waiting, sell orders waiting or nothing waiting, and SOR diverts a market order in only one of these three cases (footnote 7 of the paper).
Below $D = 1/2$, the market orders that SOR diverts are limited by the orders waiting in the dark pool, so $k'$ falls as $D$ grows.
Above $D = 1/2$, they are limited by the market orders to the lit market, so $k'$ stops falling.
The paper concludes that whether $D$ is above or below 1/2 is what matters, and that this suggests $D = 1/2$ as the optimal usage.

For the price discovery function, the paper considers the case where $D$ is well above 1/2, $P^t > P_f$ and many buy orders are waiting in the dark pool.
SOR then sends all the market sell orders to the dark pool, and the waiting buy orders never decrease if

$$a d > (1 - a) d + k (1 - a)(1 - d). \qquad (11)$$

With Eq. (4), this becomes

$$a > \frac{1 + D}{1 + 3D}. \qquad (12)$$

In this case the price cannot return to $P_f$, and the price discovery function of the lit market is lost.
At $D = 20\%$, this needs $a > 75\%$, that is, three times as many buy orders as sell orders, which the paper regards as rare.
At $D = 90\%$, a very slight imbalance of $a > 51\%$ or so is enough (Fig. 10).

### Conclusions

The paper suggests that the optimal $D$ is around 50% to 70%, much higher than the cap of 8% discussed in Europe, although it could not determine the exact level.
It also stresses that an artificial market shows possible mechanisms rather than forecasts, so its results should be compared with other kinds of studies, such as empirical ones.

### What this example can show

This example has no SOR, so it can be compared only with the case without SOR, and not with the optimal usage, Figs. 3 to 10 or the equation model of SOR.
Its parameters, order rules and scale also differ from those of the paper (see Config), so the comparison is only qualitative.
It can still check Eq. (4), the relation between $d$ and $D$, and show how $M_{ie}$ and the execution rate of the lit market change with $D$.

## Usage of the dark pool

The cells below vary `darkPoolChance` over 0, 0.1, 0.2, 0.3, 0.5, 0.7 and 0.9, a subset of the values of $d$ in the paper.
As in the paper, every value uses the same random numbers, here the seeds 1 to 10.
Session 1 is lengthened to 2,000 steps so that the measures fluctuate less, and the sweep takes some tens of seconds, depending on the machine.
This is still much shorter and fewer than the runs of the paper, $10^7$ steps repeated 100 times.

In [ ]:
dark_pool_chances = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
seeds = list(range(1, 11))
config_sweep = copy.deepcopy(config)
config_sweep["simulation"]["sessions"][1]["iterationSteps"] = 2000
summaries: Dict[float, List[Dict[str, float]]] = {}
for dark_pool_chance in dark_pool_chances:
    config_sweep["DarkPoolFCNAgents"]["darkPoolChance"] = dark_pool_chance
    summaries[dark_pool_chance] = [summarize(run(config=config_sweep, seed=seed)) for seed in seeds]

The table shows, for each $d$, the share $D$ predicted by Eq. (4), the simulated $D$, $M_{ie}$ with its standard deviation over the seeds and the execution rate of the lit market.
Eq. (4) uses the mean execution rate of the lit market at $d = 0$ as $k$.

In [ ]:
def predict_dark_share(dark_pool_chance: float, execution_rate: float) -> float:
    # Eq. (4) of Mizuta et al. (2015)
    return (dark_pool_chance / 2) / (execution_rate * (1 - dark_pool_chance) + dark_pool_chance / 2)


keys = ["dark_share", "market_inefficiency", "lit_execution_rate"]
values = {key: np.array([[x[key] for x in summaries[c]] for c in dark_pool_chances]) for key in keys}
means = {key: value.mean(axis=1) for key, value in values.items()}
stds = {key: value.std(axis=1) for key, value in values.items()}
k = float(means["lit_execution_rate"][0])

print(f"k (execution rate of the lit market at d = 0): {k:.3f}")
print("   d  D (Eq. 4)  D (simulated)  M_ie [%]         execution rate (lit)")
for i, c in enumerate(dark_pool_chances):
    print(
        f"{c:4.1f}  {predict_dark_share(c, k):9.3f}  {means['dark_share'][i]:13.3f}"
        f"  {means['market_inefficiency'][i]:6.3f} ± {stds['market_inefficiency'][i]:5.3f}"
        f"  {means['lit_execution_rate'][i]:20.3f}"
    )

The left plot shows $D$ against $d$ with Eq. (4) and the line $D = d$.
The other plots show $M_{ie}$ and the execution rate of the lit market against $D$, as Figs. 1 and 2 of the paper do.
The error bars are the standard deviations over the seeds.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
d_grid = np.linspace(0.0, 0.9, 91)
axes[0].errorbar(
    dark_pool_chances, means["dark_share"], yerr=stds["dark_share"], marker="o", capsize=3, linestyle="none", label="simulation"
)
axes[0].plot(d_grid, [predict_dark_share(c, k) for c in d_grid], label="Eq. (4)")
axes[0].plot(d_grid, d_grid, color="gray", linestyle="--", label="D = d")
axes[0].set_xlabel("d (darkPoolChance)")
axes[0].set_ylabel("D (share of the dark pool)")
axes[0].legend()
for ax, (key, label) in zip(
    axes[1:], [("market_inefficiency", "$M_{ie}$ [%]"), ("lit_execution_rate", "execution rate of the lit market")]
):
    ax.errorbar(means["dark_share"], means[key], yerr=stds[key], marker="o", capsize=3)
    ax.set_xlabel("D (share of the dark pool)")
    ax.set_ylabel(label)
plt.tight_layout()
plt.show()

### Reading the results

With these seeds, the results are as follows.

- The simulated $D$ is larger than $d$ for every $d > 0$, as Eq. (4) predicts for $k \approx 0.38 < 1/2$.
  It is close to Eq. (4) but slightly below it, for example 0.23 against 0.25 at $d = 0.2$.
- The execution rate of the lit market stays between 0.37 and 0.38 up to $d = 0.7$ and falls to about 0.36 at $d = 0.9$, by less than its standard deviation over the seeds.
  The paper also finds that the execution rate of the lit market is stable and decreases only moderately without SOR (Fig. 2).
- The mean of $M_{ie}$ rises slightly, from about 2.1% at $D = 0$ to about 2.3% at $D \approx 0.74$, by less than its standard deviation over the seeds, and jumps to about 3.8% with a large spread at $D \approx 0.92$.
  This is unlike the case without SOR in the paper, where $M_{ie}$ decreases gradually as $D$ grows (Fig. 1).
  $M_{ie}$ is also several times as large as in Fig. 1 of the paper.

The parameters, the order rules and the scale of this example differ from those of the paper (see Config), and its runs are much shorter and fewer, so these results do not reproduce the paper.
This example does not examine which of the differences causes the different behavior of $M_{ie}$.
To explore it, change the config toward the paper, for example `chartWeight`, `timeWindowSize` and `numAgents`, and use longer runs and more seeds.

## Differences from plhamJ

The classes of this example are ported from the DarkPool sample of plhamJ, and `samples/dark_pool/main.py` prints the same columns as that sample.
They differ from it as follows.

- `maxNormalOrders` is set to 2 in the config, because the default of plhamJ is the number of markets while that of PAMS is 1.
- The orders to the dark pool are market orders, while plhamJ uses limit orders with the largest price.
  The dark pool rejects limit orders.
- plhamJ ignores the orders to the dark pool while the dark pool is not running.
  The dark pool here places such orders and cancels them at once.
- plhamJ sends an order to the dark pool only if the mid price of the dark pool is not finite, which is always true.
  The condition is omitted.
- PAMS has no itayose at the start of a session, so the orders placed in the first session are executed at the first order of the second session.
- The market price of PAMS is the last executed price even at the steps without executions, while plhamJ uses the mid price at such steps.
- `FCNAgent` of PAMS uses `timeWindowSize` for the time to live of its orders and for the expected price, while that of plhamJ uses `min(t, timeWindowSize)`.
  So the agents of plhamJ do not order at t = 0, and their first orders expire sooner.